In [1]:
import os
import sys
import traceback

import pandas as pd
from joblib import Parallel, delayed

sys.path.insert(0, 'dataprep')
from species_manifest import load_species_manifest
from maxent_model import (
    INFERENCE_SEASONS,
    audit_inference_outputs,
    inference_jobs,
    inference_season_complete,
    run_inference_species,
    species_output_dir,
)


In [2]:
basedir = '/mnt/f/biodiversity'
outputdir = os.path.join(basedir, 'modelprep')
queue_csv = os.path.join('dataprep', 'inference_rerun_queue.csv')

# Parallel species workers (each worker runs all queued seasons for one species serially).
jobs = 8
run_mode = 'missing_only'  # 'missing_only' or 'all'
seasons = INFERENCE_SEASONS
force_stack = False  # set True to rebuild inference_stack_{season}.tif caches
verbose = False  # set True for band mapping / GDAL detail (debugging)

manifest = load_species_manifest()
species_list = [s.replace(' ', '_').lower() for s in manifest['scientific_name'].tolist()]

# species_list = ['anaxyrus_americanus', ...]  # optional subset override

os.makedirs(outputdir, exist_ok=True)
print(f'Loaded {len(species_list)} species from Excel manifest')


Loaded 11 species from Excel manifest


In [3]:
audit = audit_inference_outputs(species_list, basedir, seasons=seasons)
job_list = inference_jobs(
    species_list,
    basedir,
    missing_only=(run_mode == 'missing_only'),
    seasons=seasons,
)

print(f"Modelable species: {audit['modelable_species']}/{len(species_list)}")
print(f"Complete species: {len(audit['complete'])}/{audit['modelable_species']}")
print(f"Partial species: {len(audit['partial'])}")
print(f"Need training (unexpected — no model): {len(audit['need_training'])}")
print(f"MaxEnt excluded (known failures): {len(audit['maxent_excluded'])}")
print(f"Prediction tifs: {audit['present_tifs']}/{audit['expected_tifs']}")
print(f"Inference jobs queued: {len(job_list)}")

if audit['maxent_excluded']:
    print('\nExcluded from inference (MaxEnt training failed):')
    for row in audit['maxent_excluded']:
        print(f"  {row['species']}: {row['reason']}")

if audit['partial']:
    print('\nPartial species (re-run these):')
    for row in audit['partial']:
        print(f"  {row['species']}: missing {row['missing_seasons']}")

if audit['need_training']:
    print('\nUnexpected — no trained model (investigate):')
    for sp in audit['need_training']:
        print(f'  {sp}')

rows = [{'species': sp, 'season': tp, 'reason': 'missing_outputs'} for sp, tp in job_list]
pd.DataFrame(rows).to_csv(queue_csv, index=False)
print(f'\nWrote queue to {queue_csv}')
for sp, tp in job_list:
    print(f'  {sp} - {tp}')


Modelable species: 11/11
Complete species: 0/11
Partial species: 11
Need training (unexpected — no model): 0
MaxEnt excluded (known failures): 0
Prediction tifs: 432/88
Inference jobs queued: 12

Partial species (re-run these):
  anaxyrus_americanus: missing ['spring']
  anaxyrus_fowleri: missing ['spring']
  chelydra_serpentina: missing ['summer']
  cistothorus_palustris: missing ['winter']
  coccyzus_americanus: missing ['fall', 'winter']
  dryophytes_cinereus: missing ['spring']
  farancia_abacura: missing ['winter']
  gastrophryne_carolinensis: missing ['spring']
  kinosternon_subrubrum: missing ['summer']
  nerodia_rhombifer_rhombifer: missing ['spring']
  rallus_elegans: missing ['winter']

Wrote queue to dataprep/inference_rerun_queue.csv
  anaxyrus_americanus - spring
  anaxyrus_fowleri - spring
  chelydra_serpentina - summer
  cistothorus_palustris - winter
  coccyzus_americanus - fall
  coccyzus_americanus - winter
  dryophytes_cinereus - spring
  farancia_abacura - winter
  

In [4]:
def run_one_species(sp: str, species_jobs: list[tuple[str, str]]) -> list[tuple]:
    season_list = [tp for _, tp in species_jobs]
    print(f'Running {sp} ({len(season_list)} season(s))')
    try:
        results = run_inference_species(
            sp, basedir, season_list, force_stack=force_stack, verbose=verbose
        )
        spdir = species_output_dir(basedir, sp)
        out: list[tuple] = []
        for row in results:
            if not inference_season_complete(spdir, sp, row['season']):
                raise RuntimeError(f'prediction tifs missing for {row["season"]}')
            out.append((sp, row['season'], 'ok'))
        return out
    except Exception as exc:
        print(f'FAILED {sp}: {exc}')
        traceback.print_exc()
        return [(sp, '', 'fail', str(exc))]


In [5]:
if not job_list:
    print('Nothing to run.')
else:
    by_species: dict[str, list[tuple[str, str]]] = {}
    for sp, tp in job_list:
        by_species.setdefault(sp, []).append((sp, tp))

    species_order = sorted(by_species.keys())
    if jobs <= 1:
        completed = [run_one_species(sp, by_species[sp]) for sp in species_order]
    else:
        completed = Parallel(n_jobs=jobs, verbose=1)(
            delayed(run_one_species)(sp, by_species[sp]) for sp in species_order
        )
    completed = [item for group in completed for item in group]

    ok = [r for r in completed if len(r) >= 3 and r[2] == 'ok']
    failed = [r for r in completed if len(r) >= 4 and r[2] == 'fail']
    print(f'\nFinished: {len(ok)} season job(s) ok, {len(failed)} failed')
    for row in failed:
        print(f"  FAIL {row[0]}: {row[3]}")


/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running anaxyrus_americanus - spring


Executing:  62%|████████████████████████████████████████████▍                          | 5/8 [00:04<00:01,  1.78cell/s]/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(

Executing: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [01:12<00:00,  9.07s/cell]


Running anaxyrus_fowleri - spring


Executing:  75%|█████████████████████████████████████████████████████▎                 | 6/8 [00:04<00:00,  2.60cell/s]/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(

Executing: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [01:16<00:00,  9.51s/cell]


...

Running nerodia_rhombifer_rhombifer - spring


Executing:  75%|█████████████████████████████████████████████████████▎                 | 6/8 [00:04<00:00,  2.52cell/s]/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(

Executing: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [00:39<00:00,  4.98s/cell]


Running rallus_elegans - winter


Executing:  75%|█████████████████████████████████████████████████████▎                 | 6/8 [00:04<00:00,  2.63cell/s]/home/mike/miniforge3/envs/biodiversity/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(

Executing: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [01:05<00:00,  8.16s/cell]


Finished: 12 ok, 0 failed


In [6]:
audit_after = audit_inference_outputs(species_list, basedir, seasons=seasons)
print(
    f"After run — complete: {len(audit_after['complete'])}/{audit_after['modelable_species']} modelable "
    f"({len(audit_after['complete'])}/{len(species_list)} manifest)"
)
print(f"Prediction tifs: {audit_after['present_tifs']}/{audit_after['expected_tifs']}")
if audit_after['partial']:
    print('Still partial:')
    for row in audit_after['partial']:
        print(f"  {row['species']}: {row['missing_seasons']}")


After run — complete: 11/11
Prediction tifs: 456/88
